# Aula 3 - Engenharia de Dados Financeiros e Valuation

Este notebook foi reorganizado para começar pelo bloco offline de valuation e só depois entrar em coleta de dados externos.

## Ordem da aula
- Construção de um caso-base de projeção.
- Cálculo do fluxo de caixa livre da firma (`FCFF`, aqui também referido como `FICC/FCF da firma` para manter aderência à aula).
- Cálculo do WACC.
- Cálculo do VPL e da TIR.
- Sensibilidade do valuation.
- Coleta de dados externos para enriquecer o modelo.

As células de código foram comentadas para facilitar sua explicação em sala.

## Ambiente recomendado

Se o VS Code ou o Jupyter reclamar de kernel ou `ipykernel`, crie um ambiente virtual na raiz do repositório e instale os pacotes mínimos:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
python -m pip install -r pages/module-5-adm-tech/lesson-3-requirements.txt
python -m ipykernel install --user --name aulas-lesson-3 --display-name "Python (aulas lesson-3)"
```

Depois, selecione o kernel `Python (aulas lesson-3)` no notebook.

## 1. Setup geral

Primeiro importamos as bibliotecas e configuramos o ambiente. Repare que o notebook tenta usar `yfinance`, mas continua funcionando mesmo sem internet ou sem essa biblioteca.

In [ ]:
# Todos os imports necessários do notebook ficam nesta primeira célula.
# Import futuro para deixar as anotações de tipo mais previsíveis.
from __future__ import annotations

# Path será usado para salvar artefatos da aula.
from pathlib import Path

# Time será usado em uma comparação simples de performance.
import time

# NumPy para cálculos numéricos e Pandas para modelagem tabular.
import numpy as np
import pandas as pd

# display ajuda a renderizar saídas tabulares de forma mais clara no notebook.
from IPython.display import display

# Requests será usado quando chegarmos ao bloco de consulta ao Banco Central.
import requests

# Tentamos importar yfinance, mas o notebook continua mesmo se essa dependência não existir.
try:
    import yfinance as yf
except Exception:
    yf = None

# Formatadores para evitar notação científica nas tabelas financeiras.
MONEY_FMT = "R$ {:,.2f}"
RATE_FMT = "{:.2%}"

# Configurações de exibição para deixar os DataFrames mais legíveis em aula.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Semente fixa para tornar os exemplos sintéticos reproduzíveis.
np.random.seed(42)

# Pasta onde vamos salvar arquivos gerados ao longo do notebook.
DATA_DIR = Path("pages/module-5-adm-tech/lesson-3-artifacts")
DATA_DIR.mkdir(exist_ok=True)

# Horizonte de projeção do valuation e intervalo usado depois nos dados de mercado.
projection_years = np.arange(2025, 2030)
START = "2024-01-01"
END = "2025-03-01"
TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA"]

print(f"Diretório de saída: {DATA_DIR.resolve()}")
print(f"yfinance disponível? {'sim' if yf else 'não'}")

: 

## 2. Caso-base de valuation

Antes de buscar dados externos, começamos com um caso-base completamente controlado. Isso é importante didaticamente porque separa duas coisas:

- a lógica financeira do valuation;
- a qualidade e a disponibilidade dos dados.

Aqui vamos projetar receita, margem, impostos, reinvestimento e fluxo de caixa livre da firma.

In [ ]:
# Criamos um dicionário com as premissas centrais do caso.
# A ideia é deixar claro para a turma que valuation nasce de premissas explícitas.
valuation_inputs = {
    "receita_inicial": 180_000_000,
    "crescimento_receita": [0.10, 0.09, 0.08, 0.07, 0.06],
    "margem_ebit": [0.18, 0.19, 0.20, 0.205, 0.21],
    "aliquota_imposto": 0.34,
    "depreciacao_pct_receita": [0.03, 0.031, 0.031, 0.032, 0.032],
    "capex_pct_receita": [0.055, 0.052, 0.050, 0.048, 0.047],
    "delta_wc_pct_receita": [0.012, 0.011, 0.010, 0.010, 0.009],
}

# Montamos um DataFrame indexado pelos anos projetados.
proj = pd.DataFrame(index=projection_years)

# Guardamos as premissas por coluna para deixar a projeção rastreável.
proj["growth_rate"] = valuation_inputs["crescimento_receita"]
proj["ebit_margin"] = valuation_inputs["margem_ebit"]
proj["dep_pct"] = valuation_inputs["depreciacao_pct_receita"]
proj["capex_pct"] = valuation_inputs["capex_pct_receita"]
proj["delta_wc_pct"] = valuation_inputs["delta_wc_pct_receita"]

# A primeira receita projetada nasce da receita-base multiplicada pelo crescimento do primeiro ano.
proj.loc[projection_years[0], "receita"] = valuation_inputs["receita_inicial"] * (1 + proj.loc[projection_years[0], "growth_rate"])

# Os anos seguintes crescem sobre a receita do ano anterior.
for year in projection_years[1:]:
    previous_year = year - 1
    proj.loc[year, "receita"] = proj.loc[previous_year, "receita"] * (1 + proj.loc[year, "growth_rate"])

# Exibimos apenas as premissas e a receita para conferir o caso-base antes de aprofundar.
display(
    proj[["growth_rate", "ebit_margin", "dep_pct", "capex_pct", "delta_wc_pct", "receita"]].style.format(
        {
            "growth_rate": RATE_FMT,
            "ebit_margin": RATE_FMT,
            "dep_pct": RATE_FMT,
            "capex_pct": RATE_FMT,
            "delta_wc_pct": RATE_FMT,
            "receita": MONEY_FMT,
        }
    )
)

## 3. FICC / FCFF / Fluxo de Caixa Livre da Firma

Agora transformamos a projeção operacional em fluxo de caixa.

A lógica usada é:

$$FCFF = EBIT 	imes (1 - imposto) + depreciação - capex - \Delta capital\ de\ giro$$

Na prática, este é o fluxo disponível para todos os financiadores da firma, antes da divisão entre dívida e equity.

In [ ]:
# EBIT é a receita multiplicada pela margem operacional projetada.
proj["EBIT"] = proj["receita"] * proj["ebit_margin"]

# O imposto operacional incide sobre o EBIT.
proj["imposto_operacional"] = proj["EBIT"] * valuation_inputs["aliquota_imposto"]

# NOPAT é o lucro operacional após impostos.
proj["NOPAT"] = proj["EBIT"] - proj["imposto_operacional"]

# Depreciação é tratada como percentual da receita.
proj["depreciacao"] = proj["receita"] * proj["dep_pct"]

# Capex representa o investimento para sustentar ou expandir a operação.
proj["capex"] = proj["receita"] * proj["capex_pct"]

# Variação de capital de giro também é modelada como percentual da receita.
proj["delta_wc"] = proj["receita"] * proj["delta_wc_pct"]

# Aqui calculamos o fluxo de caixa livre da firma.
# Em alguns materiais de aula ele também aparece como FICC ou FCF da firma.
proj["FCFF"] = proj["NOPAT"] + proj["depreciacao"] - proj["capex"] - proj["delta_wc"]

# Selecionamos só as colunas principais para explicar o encadeamento lógico do cálculo.
display(
    proj[["receita", "EBIT", "imposto_operacional", "NOPAT", "depreciacao", "capex", "delta_wc", "FCFF"]].style.format(
        {
            "receita": MONEY_FMT,
            "EBIT": MONEY_FMT,
            "imposto_operacional": MONEY_FMT,
            "NOPAT": MONEY_FMT,
            "depreciacao": MONEY_FMT,
            "capex": MONEY_FMT,
            "delta_wc": MONEY_FMT,
            "FCFF": MONEY_FMT,
        }
    )
)

## 4. WACC

Depois do fluxo, precisamos da taxa de desconto. Para isso calculamos o WACC.

A lógica é:

- custo de capital próprio via CAPM;
- custo da dívida após imposto;
- ponderação pela estrutura de capital.


In [ ]:
# Premissas financeiras para o cálculo do WACC.
capital_structure = {
    "risk_free": 0.115,
    "market_premium": 0.075,
    "beta": 1.18,
    "cost_of_debt_pre_tax": 0.135,
    "tax_rate": valuation_inputs["aliquota_imposto"]
    "equity_weight": 0.62,
    "debt_weight": 0.38,
}

# Custo de equity pelo CAPM: rf + beta * prêmio de mercado.
cost_of_equity = capital_structure["risk_free"] + capital_structure["beta"] * capital_structure["market_premium"]

# Custo da dívida após imposto incorpora o benefício fiscal da despesa financeira.
after_tax_cost_of_debt = capital_structure["cost_of_debt_pre_tax"] * (1 - capital_structure["tax_rate"])

# WACC é a média ponderada dos custos de capital.
wacc = (
    capital_structure["equity_weight"] * cost_of_equity
    + capital_structure["debt_weight"] * after_tax_cost_of_debt
)

# Organizamos a saída em uma Series para facilitar a leitura em aula.
pd.Series(
    {
        "custo_equity": cost_of_equity,
        "custo_divida_pos_imposto": after_tax_cost_of_debt,
        "WACC": wacc,
    }
).map(lambda x: f"{x:.2%}")

## 5. VPL do fluxo de caixa

Com fluxo e taxa de desconto, calculamos o valor presente. Também vamos incluir um valor terminal via fórmula de Gordon.

A construção será:

- descontar os `FCFF` explícitos do horizonte de projeção;
- calcular o valor terminal no último ano explícito;
- trazer tudo a valor presente;
- somar para obter o `Enterprise Value`.

In [ ]:
# Função simples de VPL para uma sequência de fluxos começando no ano 1.
def npv_from_rate(rate: float, cashflows: list[float] | np.ndarray) -> float:
    # Cada fluxo é descontado por (1 + taxa) elevado ao número do período.
    return sum(cf / ((1 + rate) ** period) for period, cf in enumerate(cashflows, start=1))


# Crescimento em perpetuidade usado no valor terminal.
terminal_growth = 0.03

# Capturamos o último FCFF explícito para calcular o valor terminal.
last_fcff = proj["FCFF"] .iloc[-1]

# Fórmula de Gordon para valor terminal: FCFF do próximo ano dividido por (WACC - g).
terminal_value = last_fcff * (1 + terminal_growth) / (wacc - terminal_growth)

# Fator de desconto de cada ano da projeção explícita.
proj["discount_factor"] = [(1 / ((1 + wacc) ** i)) for i in range(1, len(proj) + 1)]

# Valor presente de cada FCFF explícito.
proj["pv_fcff"] = proj["FCFF"] * proj["discount_factor"]

# O valor terminal acontece no fim do último ano explícito e também precisa ser descontado.
pv_terminal_value = terminal_value * proj["discount_factor"] .iloc[-1]

# O Enterprise Value é a soma do valor presente dos fluxos explícitos com o valor presente do terminal.
enterprise_value = proj["pv_fcff"] .sum() + pv_terminal_value

# Organizamos as saídas principais para discussão em aula.
valuation_summary = pd.Series(
    {
        "VPL_fluxos_explicitos": proj["pv_fcff"] .sum(),
        "VPL_valor_terminal": pv_terminal_value,
        "Enterprise_Value": enterprise_value,
    }
)

display(proj[["FCFF", "discount_factor", "pv_fcff"]])
valuation_summary.map(lambda x: f"R$ {x:,.2f}")

## 6. TIR do projeto

A TIR é a taxa que zera o valor presente líquido dos fluxos do projeto.

Como o NumPy moderno não traz mais a função antiga de `irr`, vamos implementar uma versão didática por busca binária. Isso é útil inclusive para explicar o conceito em sala.

In [ ]:
# Função de VPL incluindo o fluxo no período zero.
def npv_with_initial(rate: float, cashflows: list[float] | np.ndarray) -> float:
    # O primeiro fluxo fica no tempo zero e não sofre desconto.
    return sum(cf / ((1 + rate) ** period) for period, cf in enumerate(cashflows))


# Função simples para aproximar a TIR por busca binária.
def irr_bisection(cashflows: list[float] | np.ndarray, low: float = -0.9, high: float = 1.5, tol: float = 1e-7, max_iter: int = 500) -> float:
    # Calculamos o VPL nos extremos para garantir que existe mudança de sinal.
    npv_low = npv_with_initial(low, cashflows)
    npv_high = npv_with_initial(high, cashflows)

    # Se não houver mudança de sinal, a TIR pode não estar bem definida nesse intervalo.
    if npv_low * npv_high > 0:
        raise ValueError("Nao foi possivel encontrar TIR no intervalo informado.")

    # Iteramos refinando o intervalo até convergir.
    for _ in range(max_iter):
        mid = (low + high) / 2
        npv_mid = npv_with_initial(mid, cashflows)

        # Se o VPL no ponto médio estiver muito perto de zero, terminamos.
        if abs(npv_mid) < tol:
            return mid

        # Mantemos o subintervalo onde a mudança de sinal continua existindo.
        if npv_low * npv_mid < 0:
            high = mid
            npv_high = npv_mid
        else:
            low = mid
            npv_low = npv_mid

    # Se não convergir antes, devolvemos a melhor aproximação final.
    return (low + high) / 2


# Assumimos um investimento inicial negativo no tempo zero.
initial_investment = -210_000_000

# Para a TIR, consideramos o investimento inicial e os fluxos projetados.
# No último ano somamos o valor terminal, pois ele representa a continuidade do negócio.
project_cashflows = [initial_investment] + proj["FCFF"] .tolist()
project_cashflows[-1] = project_cashflows[-1] + terminal_value

# Calculamos a TIR aproximada do projeto.
project_irr = irr_bisection(project_cashflows)

# Também calculamos o VPL desse conjunto completo de fluxos, usando o WACC, para comparação.
project_npv_at_wacc = npv_with_initial(wacc, project_cashflows)

pd.Series(
    {
        "investimento_inicial": initial_investment,
        "TIR_projeto": project_irr,
        "VPL_ao_WACC": project_npv_at_wacc,
    }
)

## 7. Sensibilidade do valuation

Em valuation, um único número costuma passar falsa precisão. Por isso montamos uma análise de sensibilidade variando `WACC` e crescimento na perpetuidade.

In [ ]:
# Definimos uma grade simples de cenários para WACC e crescimento terminal.
wacc_grid = [wacc - 0.02, wacc - 0.01, wacc, wacc + 0.01, wacc + 0.02]
growth_grid = [0.02, 0.025, 0.03, 0.035, 0.04]

# Vamos armazenar os resultados em uma tabela para leitura cruzada.
sensitivity = pd.DataFrame(index=[f"g={g:.1%}" for g in growth_grid], columns=[f"WACC={r:.1%}" for r in wacc_grid])

# Recalculamos o valor terminal e o enterprise value para cada combinação.
for g in growth_grid:
    for r in wacc_grid:
        # Evitamos combinações economicamente inconsistentes em que g >= WACC.
        if g >= r:
            sensitivity.loc[f"g={g:.1%}", f"WACC={r:.1%}"] = np.nan
            continue

        # Descontamos os fluxos explícitos com a taxa do cenário.
        explicit_value = npv_from_rate(r, proj["FCFF"] .tolist())

        # Recalculamos o valor terminal do cenário.
        tv = proj["FCFF"] .iloc[-1] * (1 + g) / (r - g)
        pv_tv = tv / ((1 + r) ** len(proj))

        # O valor final do cenário é a soma do explícito com o terminal descontado.
        sensitivity.loc[f"g={g:.1%}", f"WACC={r:.1%}"] = explicit_value + pv_tv

# Convertendo para float para melhorar a visualização.
sensitivity = sensitivity.astype(float)

# Em pandas mais novo, usamos DataFrame.map para formatar célula a célula.
sensitivity.map(lambda x: f"R$ {x/1_000_000:,.1f} mi" if pd.notnull(x) else "-")

## 8. Leituras rápidas para a aula

Até aqui já temos um bloco completo de valuation sem depender da internet. Isso permite explicar:

- como premissas viram fluxo;
- como fluxo vira valor;
- como WACC e crescimento terminal alteram fortemente o valuation;
- por que TIR e VPL respondem perguntas parecidas, mas não idênticas.

Só agora vamos para a parte de dados externos, que entra como incremento e não como pré-requisito para começar a aula.

## 9. Coleta de preços de mercado

Agora entramos na camada de dados. A função abaixo tenta baixar preços reais via Yahoo Finance. Se isso falhar, ela cria uma base sintética para manter a prática executável.

In [ ]:
# Esta função gera preços sintéticos com estrutura parecida com a de um provedor real.
def generate_synthetic_prices(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    # Usamos frequência de dias úteis porque o mercado não negocia em todos os dias corridos.
    dates = pd.date_range(start, end, freq="B")

    # As colunas imitam um dataset OHLCV clássico.
    fields = ["Open", "High", "Low", "Close", "Volume"]
    columns = pd.MultiIndex.from_product([fields, tickers], names=["Field", "Ticker"])
    data = pd.DataFrame(index=dates, columns=columns, dtype=float)

    # Criamos uma trajetória sintética para cada ativo.
    for ticker in tickers:
        # Definimos um preço-base inicial aleatório para cada ticker.
        base_price = np.random.uniform(20, 60)

        # Geramos pequenas variações diárias com tendência e ruído.
        noise = np.random.normal(0.0004, 0.018, len(dates))
        close = base_price * np.exp(np.cumsum(noise))

        # Montamos as outras colunas ao redor do fechamento.
        open_ = close * (1 + np.random.normal(0, 0.003, len(dates)))
        high = np.maximum(open_, close) * (1 + np.random.uniform(0.001, 0.02, len(dates)))
        low = np.minimum(open_, close) * (1 - np.random.uniform(0.001, 0.02, len(dates)))
        volume = np.random.randint(800_000, 8_000_000, len(dates))

        # Gravamos as séries no DataFrame final.
        data[("Open", ticker)] = open_
        data[("High", ticker)] = high
        data[("Low", ticker)] = low
        data[("Close", ticker)] = close
        data[("Volume", ticker)] = volume

    # Ordenamos as colunas para manter consistência na leitura.
    return data.sort_index(axis=1)


# Esta função tenta primeiro a coleta real e cai para dados sintéticos em caso de falha.
def load_market_data(tickers: list[str], start: str, end: str) -> tuple[pd.DataFrame, str]:
    # Só tentamos a coleta real se a biblioteca existir.
    if yf is not None:
        try:
            # yfinance já devolve um formato conveniente para múltiplos ativos.
            market = yf.download(
                tickers=tickers,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                group_by="column",
                threads=True,
            )

            # Se vier dado, padronizamos os nomes dos níveis do MultiIndex.
            if not market.empty:
                market.columns.names = ["Field", "Ticker"]
                return market.sort_index(axis=1), "yfinance"
        except Exception as exc:
            # Mantemos a mensagem para explicar em aula o tratamento de falhas externas.
            print(f"Falha na coleta real: {exc}")

    # Se a coleta real não funcionar, retornamos a base simulada.
    return generate_synthetic_prices(tickers, start, end), "sintetico"


# Carregamos a base de mercado que será usada nos próximos exemplos.
market_data, market_source = load_market_data(TICKERS, START, END)
print(f"Fonte usada: {market_source}")
market_data.head()

In [ ]:
# Extraímos preços de fechamento e volume para trabalhar com tabelas mais simples.
close_prices = market_data["Close"] .copy()
volume_data = market_data["Volume"] .copy()

# Exibimos dimensões e um pequeno resumo para entender o dataset carregado.
print("Dimensão do dataset de preços:", market_data.shape)
print("Tickers carregados:", list(close_prices.columns))
display(close_prices.tail())
display(volume_data.describe().T)

## 10. Consulta ao Banco Central

Agora adicionamos o contexto macroeconômico. A ideia é combinar preços de mercado com séries como Selic e IPCA.

In [ ]:
# Mapeamos os códigos do SGS para nomes mais legíveis.
SERIES_MAP = {
    432: "selic",
    433: "ipca",
}


# Esta função consulta uma série específica na API do Banco Central.
def fetch_bcb_series(series_code: int, start: str, end: str) -> pd.Series:
    # Montamos a URL usando o código da série.
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{series_code}/dados"

    # O formato da API do BCB usa datas no padrão dia/mês/ano.
    params = {
        "formato": "json",
        "dataInicial": pd.Timestamp(start).strftime("%d/%m/%Y"),
        "dataFinal": pd.Timestamp(end).strftime("%d/%m/%Y"),
    }

    # Fazemos a chamada HTTP e levantamos erro se a resposta vier ruim.
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()

    # Transformamos a resposta em DataFrame para tratar datas e valores.
    payload = pd.DataFrame(response.json())
    payload["data"] = pd.to_datetime(payload["data"], dayfirst=True)
    payload["valor"] = pd.to_numeric(payload["valor"], errors="coerce")

    # Retornamos a série indexada por data.
    return payload.set_index("data")["valor"]


# Função que reúne as séries macroeconômicas de interesse.
def load_macro_data(start: str, end: str) -> tuple[pd.DataFrame, str]:
    try:
        # Consultamos cada série e consolidamos tudo em um único DataFrame.
        macro = {
            name: fetch_bcb_series(code, start, end)
            for code, name in SERIES_MAP.items()
        }
        frame = pd.concat(macro, axis=1).sort_index()
        return frame, "BCB"
    except Exception as exc:
        # Se a consulta falhar, criamos séries sintéticas para não travar a aula.
        print(f"Falha ao consultar o BCB: {exc}")
        idx = pd.date_range(start, end, freq="B")
        synthetic = pd.DataFrame(index=idx)
        synthetic["selic"] = 10.5 + np.sin(np.linspace(0, 3, len(idx))) * 0.35
        synthetic["ipca"] = 0.25 + np.sin(np.linspace(0, 6, len(idx))) * 0.08
        return synthetic, "sintetico"


# Carregamos as séries macro para usar na integração dos dados.
macro_data, macro_source = load_macro_data(START, END)
print(f"Fonte macro usada: {macro_source}")
macro_data.head()

## 11. Integração das bases

Aqui juntamos mercado e macroeconomia. Esse passo é central porque muitos erros em análise financeira nascem de alinhamento incorreto de datas.

In [ ]:
# Calculamos retornos percentuais diários a partir dos fechamentos.
returns = close_prices.pct_change().add_suffix("_ret")

# Renomeamos os preços de fechamento para ficar claro, depois do merge, o que é preço e o que é retorno.
prices_flat = close_prices.add_suffix("_close")

# Juntamos preços, retornos e séries macro em uma mesma base analítica.
analytics_base = prices_flat.join(returns).join(macro_data, how="left")
analytics_base = analytics_base.sort_index()

# Preenchemos Selic e IPCA para frente porque séries macro podem não atualizar em todos os dias úteis do mercado.
analytics_base[["selic", "ipca"]] = analytics_base[["selic", "ipca"]] .ffill()

# Conferimos o resultado da integração.
display(analytics_base.head())
print("Valores nulos por coluna:")
analytics_base.isna() .sum()

In [ ]:
# Definimos os caminhos dos arquivos que serão gerados.
csv_path = DATA_DIR / "dados_financeiros_tratados.csv"
parquet_path = DATA_DIR / "dados_financeiros_tratados.parquet"

# Salvamos em CSV porque é o formato mais universal para inspeção rápida.
analytics_base.to_csv(csv_path, index=True)
print(f"CSV salvo em: {csv_path}")

# Tentamos salvar em Parquet porque ele é mais eficiente para reuso analítico.
try:
    analytics_base.to_parquet(parquet_path, index=True)
    print(f"Parquet salvo em: {parquet_path}")
except Exception as exc:
    # Se pyarrow não estiver disponível, o notebook continua normalmente.
    print(f"Parquet não foi salvo: {exc}")

## 12. Séries temporais e indicadores

Agora usamos uma ação como exemplo para mostrar alguns cálculos clássicos: retornos, médias móveis, volatilidade e agregações temporais.

In [ ]:
# Criamos um DataFrame só para PETR4.SA, o que facilita a explicação em aula.
petr = pd.DataFrame(index=market_data.index)

# Selecionamos as colunas relevantes da ação.
petr["Close"] = market_data[("Close", "PETR4.SA"])
petr["High"] = market_data[("High", "PETR4.SA"])
petr["Low"] = market_data[("Low", "PETR4.SA"])
petr["Volume"] = market_data[("Volume", "PETR4.SA"])

# Retorno simples mede a variação percentual entre um dia e o seguinte.
petr["simple_return"] = petr["Close"] .pct_change()

# Log-return é útil em vários contextos quantitativos, especialmente composição e modelagem.
petr["log_return"] = np.log(petr["Close"] / petr["Close"] .shift(1))

# Médias móveis ajudam a resumir tendência recente.
petr["SMA_20"] = petr["Close"] .rolling(20) .mean()
petr["SMA_50"] = petr["Close"] .rolling(50) .mean()
petr["EMA_20"] = petr["Close"] .ewm(span=20, adjust=False) .mean()

# Volatilidade anualizada usando janela móvel de 21 dias úteis.
petr["vol_21d"] = petr["simple_return"] .rolling(21) .std() * np.sqrt(252)

# Exemplo de função customizada em rolling: amplitude de preços na janela.
petr["rolling_range_20d"] = petr["Close"] .rolling(20) .apply(lambda x: x.max() - x.min(), raw=False)

# Retorno acumulado mostra a trajetória do investimento ao longo do tempo.
petr["cum_return"] = (1 + petr["simple_return"] .fillna(0)) .cumprod() - 1

# Resampling semanal para resumir o comportamento em frequência menor.
weekly_ohlc = petr[["Close", "High", "Low", "Volume"]] .resample("W") .agg({
    "Close": "last",
    "High": "max",
    "Low": "min",
    "Volume": "sum",
})

# Fechamento de fim de mês como outro exemplo de mudança de frequência.
monthly_close = petr["Close"] .resample("ME") .last()

display(petr.tail())
display(weekly_ohlc.tail())
display(monthly_close.tail())

In [ ]:
# Matriz de retornos diários para comparar ativos entre si.
returns_matrix = close_prices.pct_change() .dropna()

# Correlação mostra o quanto os ativos se movem juntos.
corr_matrix = returns_matrix.corr()

# Correlação móvel ajuda a mostrar que a relação entre ativos muda no tempo.
rolling_corr = returns_matrix["PETR4.SA"] .rolling(30) .corr(returns_matrix["VALE3.SA"])

# Sharpe anualizado simplificado, assumindo taxa livre de risco zero para fins didáticos.
sharpe = (returns_matrix.mean() / returns_matrix.std()) * np.sqrt(252)

print("Matriz de correlação:")
display(corr_matrix)

print("Sharpe anualizado por ativo:")
display(sharpe.sort_values(ascending=False) .to_frame("Sharpe"))

rolling_corr.tail()

## 13. MultiIndex na prática

O próprio retorno do `yfinance` para vários ativos já é um bom exemplo de `MultiIndex`: um nível representa a métrica e o outro representa o ticker.

In [ ]:
# Mostramos os nomes dos níveis do MultiIndex para reforçar a ideia de hierarquia nas colunas.
print("Níveis das colunas:", market_data.columns.names)

# Seleção de uma coluna específica usando tupla (métrica, ticker).
print("Exemplo de seleção de PETR4.SA / Close:")
display(market_data.loc[:, ("Close", "PETR4.SA")] .tail())

# Agora criamos um MultiIndex manual, desta vez nas linhas.
sector_index = pd.MultiIndex.from_tuples(
    [
        ("Energia", "PETR4.SA"),
        ("Mineracao", "VALE3.SA"),
        ("Bancos", "ITUB4.SA"),
    ],
    names=["Setor", "Ticker"]
)

# Montamos uma pequena visão de carteira para mostrar o uso prático do índice hierárquico.
portfolio_view = pd.DataFrame(
    {
        "peso": [0.40, 0.35, 0.25],
        "retorno_esperado": returns_matrix.mean() .reindex(TICKERS) .values * 252,
        "volatilidade": returns_matrix.std() .reindex(TICKERS) .values * np.sqrt(252),
    },
    index=sector_index,
)

# Exibimos a carteira inteira, um único setor e uma agregação por nível do índice.
display(portfolio_view)
display(portfolio_view.loc["Energia"])
display(portfolio_view.groupby(level="Setor") .mean(numeric_only=True))

## 14. Mini-backtest com cruzamento de médias

A ideia aqui é construir um exemplo simples de regra sistemática. Não é um modelo de trading profissional; é um exemplo didático de como transformar indicadores em decisão.

In [ ]:
# Pegamos apenas as colunas necessárias para o sinal.
signals = petr[["Close", "simple_return", "SMA_20", "SMA_50"]] .copy()

# Removemos linhas iniciais em que as médias ainda não existem.
signals = signals.dropna() .copy()

# Regra de posição: comprado quando a média curta está acima da média longa.
signals["position"] = np.where(signals["SMA_20"] > signals["SMA_50"] , 1, 0)

# Mudança de sinal ajuda a contar entradas e saídas.
signals["signal_change"] = signals["position"] .diff() .fillna(0)

# O retorno da estratégia usa a posição do dia anterior para evitar look-ahead bias básico.
signals["strategy_return"] = signals["position"] .shift(1) .fillna(0) * signals["simple_return"]

# Construímos as curvas acumuladas para buy and hold e para a estratégia.
signals["buy_hold"] = (1 + signals["simple_return"] .fillna(0)) .cumprod() - 1
signals["strategy_curve"] = (1 + signals["strategy_return"] .fillna(0)) .cumprod() - 1

# Resumo simples dos resultados.
summary = pd.Series(
    {
        "buy_hold_total_return": signals["buy_hold"] .iloc[-1] - 1,
        "strategy_total_return": signals["strategy_curve"] .iloc[-1] - 1,
        "number_of_trades": int((signals["signal_change"] != 0) .sum()),
    }
)

display(signals.tail())
summary

## 15. Vetorização e performance

Fechamos com uma comparação simples entre operação vetorizada e loop Python. Isso ajuda a justificar por que Pandas e NumPy são tão usados em análise de dados.

In [ ]:
# Criamos uma amostra grande o suficiente para perceber diferença de desempenho.
big_sample = pd.DataFrame({
    "price": np.random.lognormal(mean=4.0, sigma=0.2, size=250_000),
    "qty": np.random.randint(1, 25, size=250_000),
})

# Primeiro medimos a abordagem vetorizada.
t0 = time.perf_counter()
big_sample["gross_vectorized"] = big_sample["price"] * big_sample["qty"]
vectorized_time = time.perf_counter() - t0

# Depois medimos um loop explícito, que costuma ser mais lento em Python puro.
t0 = time.perf_counter()
gross_loop = []
for row in big_sample[["price", "qty"]] .itertuples(index=False):
    gross_loop.append(row.price * row.qty)
loop_time = time.perf_counter() - t0

# Comparamos os tempos para mostrar a diferença de abordagem.
pd.Series(
    {
        "tempo_vetorizado_s": vectorized_time,
        "tempo_loop_s": loop_time,
        "loop_dividido_por_vetorizado": loop_time / max(vectorized_time, 1e-9),
    }
)

## 16. Atividade final: valuation a partir de um CSV

Agora a proposta é que o aluno faça sozinho um valuation completo a partir da leitura de um arquivo `.csv` com `pandas`.

### Estrutura esperada do CSV

O arquivo deve ter uma linha por ano projetado e, no mínimo, as seguintes colunas:

- `ano`
- `receita`
- `margem_ebit`
- `aliquota_imposto`
- `depreciacao_pct_receita`
- `capex_pct_receita`
- `delta_wc_pct_receita`

Exemplo de estrutura:

```csv
ano,receita,margem_ebit,aliquota_imposto,depreciacao_pct_receita,capex_pct_receita,delta_wc_pct_receita
2025,198000000,0.18,0.34,0.03,0.055,0.012
2026,215820000,0.19,0.34,0.031,0.052,0.011
2027,233085600,0.20,0.34,0.031,0.050,0.010
2028,249401592,0.205,0.34,0.032,0.048,0.010
2029,264365688,0.21,0.34,0.032,0.047,0.009
```

### Passo a passo esperado

1. Ler o arquivo com `pd.read_csv(...)`.
2. Transformar o conteúdo em um `DataFrame` indexado por `ano`.
3. Conferir tipos, valores nulos e consistência das colunas.
4. Calcular `EBIT`, imposto operacional, `NOPAT`, depreciação, `capex`, `delta_wc` e `FCFF`.
5. Definir premissas de `WACC` e crescimento em perpetuidade.
6. Calcular valor presente dos fluxos explícitos.
7. Calcular valor terminal.
8. Encontrar o `Enterprise Value`.
9. Se desejar, incluir investimento inicial e calcular `VPL` e `TIR`.
10. Apresentar uma tabela final formatada com os principais resultados.

### Entrega esperada

Ao final do notebook, o aluno deve produzir:

- a leitura do CSV com `pandas`;
- o `DataFrame` tratado;
- os cálculos intermediários do valuation;
- o valuation final feito de forma independente.

A ideia aqui é não copiar o caso anterior, mas reconstruir a lógica a partir da base lida do arquivo.

In [ ]:
# Atividade final: complete o valuation a partir de um CSV.
# Substitua o caminho abaixo pelo arquivo fornecido para a atividade.
csv_path = DATA_DIR / "atividade_valuation.csv"

# 1. Leia o arquivo CSV com pandas.
atividade_df = pd.read_csv(csv_path)

# 2. Transforme a coluna de ano em índice.
atividade_df = atividade_df.set_index("ano")

# 3. Inspecione a base.
display(atividade_df.head())
display(atividade_df.dtypes)

# 4. A partir daqui, faça sozinho:
# - EBIT
# - imposto operacional
# - NOPAT
# - depreciação
# - capex
# - delta_wc
# - FCFF
# - WACC
# - valor terminal
# - Enterprise Value
# - VPL e TIR, se aplicável

# Escreva sua solução nas próximas células.
atividade_df